In [ ]:
import numpy as np
import pandas as pd
import os
# os.chdir(r"C:\Users\darves-bornoz\Documents\Spike-sorting\analyses_neuronales")
os.chdir("/home/aube/Documents/GitHub/stim_single-unit")
import functions_Rasters_DF as fct
import pynapple as nap
import matplotlib.pyplot as plt 

root_DD = 'D:/'
root_local = 'C:/Users/darves-bornoz/Documents/'
root = root_DD
root_path = '/media/aube/Aube2/'

# big_df_resp = pd.read_excel('general_summary_updated_responsiveness_bootstrap.xlsx') # il y a responsiveness_bootstrap + ind_stim

big_df = pd.read_excel(root_path + 'Spike-sorting/Tables/general_summary_all_sessions.xlsx')
# big_df_resp = pd.read_excel('/home/aube/Documents/article_neuronal_stimic/prelim/poster_SFN/analyses_prelim_SFN/general_summary_updated_responsiveness_bootstrap.xlsx') 
# il y a responsiveness_bootstrap + ind_stim

#### TEMPORAIRE ####
big_df_resp = big_df.copy() # temp, pr pas remplacer le nom partout

# temporaire : fusion de Ant. Hippocampus et Post. Hippocampus en Hippocampus
def antpostHippo2Hippo(df,colname):
    df[colname] = ['Hippocampus' if val.endswith('Hippocampus') else val for val in df[colname].tolist()]
    return df
big_df = antpostHippo2Hippo(big_df,'loca_tt')
big_df = antpostHippo2Hippo(big_df,'loca_tt_noLat')

# Ajout de la variable elec_noLat (electrode sans la lateralité)
big_df["elec_noLat"] = (
    big_df["stim_label"]
    .str.extract(r"^([A-Za-z']+)", expand=False)  # tout avant les chiffres
    .str.rstrip("p'"))  # enlève p ou ' en fin

# Ajout de log10 des metriques de reponses :
list_metrics = ['log_ratio', 'zscore_pre', 'modulation_index']
for metric in list_metrics:
    big_df[f"log10_{metric}"] = np.log10(big_df[metric])


# Ajout variable loca_tt_anat_ZE avec une valeur par combi de loca_anat_noLat et de ZE :
loca_tt_anat_ZE = []
for index, row in big_df_resp.iterrows():
    loc = row['loca_tt_noLat'] + '_' + ['InZE' if row['tt_in_ZE'] else 'OutZE'][0] 
    loca_tt_anat_ZE.append(loc)
big_df_resp['loca_tt_anat_ZE'] = loca_tt_anat_ZE 


In [ ]:
from scipy.ndimage import gaussian_filter1d

class trial_neuron_stim():  
    
    def __init__(self, patient, session, ind_neuron, stim_index, window=(-10, 10), sigma_ms=50, truncate=4):
        self.patient = patient
        self.session = session
        self.ind_neuron = ind_neuron
        self.stim_index = int(stim_index)
        self.window = window
        mask = (big_df_resp['patient']==patient) & (big_df_resp['session'].str.endswith(session)) & (big_df_resp['clu']==ind_neuron)
        self.sameElec = big_df_resp[mask]['sameElec'].tolist()[stim_index]
        self.loca_tt_anat_ZE = big_df_resp[mask]['loca_tt_anat_ZE'].tolist()[stim_index]
        self.distance_semi_quali = big_df_resp[mask]['distance_semi_quali'].tolist()[stim_index]
        self.distance_cm = big_df_resp[mask]['distance_tt_stim'].tolist()[stim_index]
                
        # ==== paramètres de lissage (pour anticiper le bleed) ====
        self.sigma_ms = sigma_ms
        self.truncate = truncate
        self.bin_size = 0.001  # par ex. 1 ms; sinon remplacer par la vraie valeur plus bas
        bleed_time = truncate * (sigma_ms / 1000)  # durée en secondes

        path_folder = root + f'Spike-sorting/Data_folders/{patient}/{patient}_stim{session}/'
        sr = fct.get_SR(patient)
        spikes = fct.get_nwb(patient, session)

        self.stim = fct.get_stims(patient, session, root).iloc[self.stim_index, :].to_dict()
        mapping_anat = pd.read_csv(root + f'Spike-sorting/Data_folders/{patient}/mapping_anat_{patient}.txt', sep=',', engine='python')

        # dead periods :
        dict_clu2tt = fct.get_dict_tetrodeName_from_tetrodeIndex(spikes, mapping_anat)
        deadfile_elec = fct.merge_overlapping_events(pd.read_csv(path_folder + f'derivatives/{patient}_stim{session}_deadfile_{dict_clu2tt[ind_neuron][:-1]}_in_ts.txt', header=None, sep='\t') / sr)
        
        # Extraire les périodes "dead" dans la fenêtre ±window (extraction étendue systematiquement pour anticiper un bleed potentiel)
        window_bleed = (window[0] - bleed_time, window[1])
        dead_pre = deadfile_elec[(deadfile_elec[1] >= self.stim['t']+window_bleed[0]) & (deadfile_elec[0] <= self.stim['t'])]
        dead_post = deadfile_elec[(deadfile_elec[1] >= self.stim['t + durée']) & (deadfile_elec[0] <= self.stim['t + durée']+window[1])]
        
        # Clamping aux bornes de la fenêtre : uniquement fin de dead_pre
        for _, row in dead_pre.iterrows():
            row[1] = min(row[1], self.stim['t'])
        
        # Si la dernière dead_pre touche t=0 → décaler la période pré
        if len(dead_pre) > 0:
            last_dead_start = dead_pre.iloc[-1, 0]
            last_dead_end = dead_pre.iloc[-1, 1]
            if last_dead_end >= self.stim['t'] - 1e-2:
                # Décaler la fin de la fenêtre pré pour éliminer la dead juste avant t0
                # On change la borne haute de l’intervalle pré :
                pre_end_corrected = last_dead_start
                shift = self.stim['t'] - last_dead_start
            else:
                pre_end_corrected = self.stim['t']
                shift = 0
        else:
            pre_end_corrected = self.stim['t']        
            shift = 0

        # Clamping bis : 
        for _, row in dead_pre.iterrows():
            row[0] = max(pre_end_corrected+window_bleed[0], row[0])
            row[1] = min(row[1], pre_end_corrected)
        for _, row in dead_post.iterrows():
            row[0] = max(self.stim['t + durée'], row[0])
            row[1] = min(row[1], self.stim['t + durée']+self.window[1])
        self.dead_pre = dead_pre
        self.dead_post = dead_post

        # spike times : 
        epoch_stim_i = nap.IntervalSet(
            start=[self.stim['t']+window_bleed[0]-shift, self.stim['t + durée']], 
            end=[pre_end_corrected, self.stim['t + durée']+window[1]], 
            time_units="s") 
        spikes_restricted_i = spikes[ind_neuron].restrict(epoch_stim_i)
        self.spike_times_raw = list(spikes_restricted_i.index)
        self.spike_times_pre = [t - self.stim['t'] for t in self.spike_times_raw if t <= self.stim['t']]
        self.spike_times_post =  [t - self.stim['durée'] - self.stim['t'] for t in self.spike_times_raw if t > self.stim['t']]

    def psth(self, bin_size=0.01, dead_value = np.nan):
        """
        bin_size : taille des bins en secondes
        dead_value : valeur pour remplir les dead periods : NaN, zero, moyenne...
        Retourne : time_bins, FR_matrix (n_trials x n_bins)
        """
        self.bin_size = bin_size # sera utilisé dans d'autres methodes
        bins = np.arange(self.window[0], self.window[1] + bin_size, bin_size)
        time_bins = bins[:-1] + bin_size / 2  # centre des bins
        
        spike_times = np.append(self.spike_times_pre, self.spike_times_post) # 1️⃣ Récupère les spikes centrés sur la stimulation
        counts, _ = np.histogram(spike_times, bins=bins) # 2️⃣ Compte des spikes par bin
        fr = counts / bin_size  # nombre -> Hz

        nan_mask = np.zeros_like(fr, dtype=bool) # 3️⃣ Crée un masque NaN pour les périodes "mortes" : rempli de False par defaut

        def mask_dead_periods(dead_df, offset=0):  # Helper interne : marquer les bins NaN
            if len(dead_df) == 0:  # pas de dead period
                return
            for _, row in dead_df.iterrows(): 
                # on centre aussi les périodes sur la stimulation
                start, end = row[0] - offset, row[1] - offset  
                nan_mask[(time_bins >= start) & (time_bins < end)] = True

        # 4️⃣ Appliquer le masquage sur les dead zones pré et post
        mask_dead_periods(self.dead_pre, offset=self.stim["t"])
        mask_dead_periods(self.dead_post, offset=self.stim["t + durée"])

        # Deux versions : dead periods en NaN et en 0
        fr_dead_NaN = np.where(nan_mask, np.nan, fr)
        fr_dead_zero = np.where(nan_mask, 0.0, fr)

        self.time_bins = time_bins
        self.fr_dead_NaN = fr_dead_NaN
        self.fr_dead_zero = fr_dead_zero
        return time_bins, fr_dead_NaN, fr_dead_zero
    
    def smooth_GC_firing_rate(self, sigma_ms=50, truncate=4):
        """
        Applique un lissage gaussien (σ en ms), puis 'shift' la période pré
        pour éliminer la contamination du post. Met à jour self.dead_pre en conséquence.
        """
        sigma_bins = (sigma_ms / 1000) / self.bin_size
        bleed_time = truncate * (sigma_ms / 1000)
        bleed_bins = int(np.ceil(bleed_time / self.bin_size))

        # --- Lissage pour les deux signaux ---
        def smooth_and_shift(fr_input):
            fr_smooth = gaussian_filter1d(fr_input, sigma=sigma_bins, truncate=truncate)
            tb = self.time_bins
            pre_idx = np.where(tb < 0)[0]
            if len(pre_idx) > bleed_bins:
                pre = fr_smooth[pre_idx].copy()
                new_pre = np.full_like(pre, np.nan)
                new_pre[bleed_bins:] = pre[:-bleed_bins]
                fr_smooth[pre_idx] = new_pre
            return fr_smooth
        
        # Création des deux lissages
        self.fr_smooth_GC = smooth_and_shift(self.fr_dead_NaN)
        self.fr_smooth_GC_dead_zero = smooth_and_shift(self.fr_dead_zero)

        # --- Ajustement des dead_pre : a decaler aussi, mais uniquement apres avoir généré les lissages
        if self.dead_pre is not None and not self.dead_pre.empty:
            shifted_dead_pre = self.dead_pre.copy()
            # Décale tout de bleed_time secondes
            shifted_dead_pre[0] += bleed_time
            shifted_dead_pre[1] += bleed_time

            # Clamp à la fenêtre valide
            t_min = self.stim["t"] + self.window[0]
            t_max = self.stim["t"]
            shifted_dead_pre[0] = np.clip(shifted_dead_pre[0], t_min, t_max)
            shifted_dead_pre[1] = np.clip(shifted_dead_pre[1], t_min, t_max)
            shifted_dead_pre = shifted_dead_pre[shifted_dead_pre[0] < shifted_dead_pre[1]]

            self.dead_pre = shifted_dead_pre.reset_index(drop=True)
    
    ### Méthodes pour bootstrap par trial : 

    def _sliding_mean(self, x, win_bins):
        """Moyenne glissante (nan-safe). Renvoie un vecteur de taille len(x)-win_bins+1."""
        if win_bins <= 1:
            return np.array([np.nanmean(x)])
        cumsum = np.nancumsum(np.insert(x, 0, 0.0))
        # nb valides glissantes
        valid = (~np.isnan(x)).astype(float)
        v_cum = np.cumsum(np.insert(valid, 0, 0.0))
        sums = cumsum[win_bins:] - cumsum[:-win_bins]
        counts = v_cum[win_bins:] - v_cum[:-win_bins]
        with np.errstate(invalid='ignore', divide='ignore'):
            m = sums / np.maximum(counts, 1.0)
        m[counts < max(1, int(0.5*win_bins))] = np.nan  # exige ≥50% d'échantillons valides
        return m

    def _sample_moving_blocks(self, base_vec, out_len, block_len):
        """
        Recompose un signal de longueur out_len en empilant des blocs contigus
        tirés aléatoirement dans base_vec (en ignorant les NaN au mieux).
        """
        x = base_vec.copy()
        # indices de départ possibles pour des blocs 'propres'
        ok = ~np.isnan(x)
        starts = []
        run = 0
        for i, v in enumerate(ok):
            run = run + 1 if v else 0
            if run >= block_len:
                starts.append(i - block_len + 1)
        if len(starts) == 0:
            return np.full(out_len, np.nan)

        out = np.empty(out_len)
        out[:] = np.nan
        i = 0
        while i < out_len:
            s = np.random.choice(starts)
            chunk = x[s:s+block_len]
            take = min(block_len, out_len - i)
            out[i:i+take] = chunk[:take]
            i += take
        return out

    def bootstrap_responsiveness(
        self,
        baseline_window=(-5.0, -1.0),
        response_window=(0.0, 4.0),
        n_boot=2000,
        min_duration=0.10,     # s : durée minimale de modulation
        block_size=0.05,       # s : taille des blocs pour le bootstrap
        alpha=0.05,
        tail='both',           # 'both', 'pos', 'neg'
        use_dead_zero=False    # True pour utiliser la version "dead=0"
    ):
        """
        Bootstrap par blocs sur FRz(t) (fr_smooth_GC_norm) d'un trial.
        Définit: self.bootstrap_p, self.bootstrap_effect, self.responsive, self.direction.
        """
        # ---- 1) Sélection du signal FRz(t) et masques temporels
        if not hasattr(self, 'fr_smooth_GC_norm'):
            raise RuntimeError("fr_smooth_GC_norm absent. Appelle normalize_and_detect_responsiveness() au moins une fois pour remplir fr_smooth_GC_norm (la normalisation neuronale).")

        z = self.fr_smooth_GC_norm_dead_zero if use_dead_zero and hasattr(self, 'fr_smooth_GC_norm_dead_zero') else self.fr_smooth_GC_norm
        t = self.time_bins
        mask_pre  = (t >= baseline_window[0]) & (t < baseline_window[1])
        mask_post = (t >= response_window[0]) & (t < response_window[1])

        z_pre  = z[mask_pre]
        z_post = z[mask_post]

        # Nettoyage si trop peu de données
        if np.sum(~np.isnan(z_pre)) < 10 or np.sum(~np.isnan(z_post)) < 5:
            self.bootstrap_p = np.nan
            self.bootstrap_effect = np.nan
            self.responsive = False
            self.direction = None
            self.bootstrap_meta = {"reason": "not_enough_valid_samples"}
            return np.nan

        # ---- 2) Statistique observée : max absolu de la moyenne glissante sur la fenêtre post
        win_bins = max(1, int(round(min_duration / self.bin_size)))
        m_post = self._sliding_mean(z_post, win_bins)
        # direction
        obs_max = np.nanmax(m_post)
        obs_min = np.nanmin(m_post)
        if tail == 'pos':
            obs = obs_max
        elif tail == 'neg':
            obs = -obs_min
        else:  # both
            obs = np.nanmax(np.abs([obs_max, obs_min]))

        # ---- 3) Nulle par moving-block bootstrap depuis la baseline
        out_len = np.sum(mask_post)
        block_bins = max(1, int(round(block_size / self.bin_size)))
        null_stats = np.empty(n_boot)
        null_stats[:] = np.nan
        for b in range(n_boot):
            synth_post = self._sample_moving_blocks(z_pre, out_len, block_bins)
            m = self._sliding_mean(synth_post, win_bins)
            if tail == 'pos':
                null_stats[b] = np.nanmax(m)
            elif tail == 'neg':
                null_stats[b] = -np.nanmin(m)
            else:
                null_stats[b] = np.nanmax(np.abs(np.array([np.nanmax(m), np.nanmin(m)])))

        # ---- 4) p-value et décision
        null_stats = null_stats[~np.isnan(null_stats)]
        if null_stats.size == 0:
            self.bootstrap_p = np.nan
            self.bootstrap_effect = np.nan
            self.responsive = False
            self.direction = None
            self.bootstrap_meta = {"reason": "null_empty"}
            return np.nan

        # +1 pour éviter p=0
        p = (np.sum(null_stats >= obs) + 1.0) / (null_stats.size + 1.0)

        # direction (si two-tailed on reporte le signe dominant observé)
        if tail == 'both':
            direction = "enhancement" if obs_max >= abs(obs_min) else "suppression"
        elif tail == 'pos':
            direction = "enhancement"
        else:
            direction = "suppression"

        self.bootstrap_p = float(p)
        self.bootstrap_effect = float(obs)        # effet (stat)
        self.responsive = bool(p < alpha)
        self.direction = direction if self.responsive else None
        self.bootstrap_meta = {
            "tail": tail,
            "alpha": alpha,
            "win_bins": int(win_bins),
            "block_bins": int(block_bins),
            "baseline_window": baseline_window,
            "response_window": response_window,
            "use_dead_zero": use_dead_zero,
            "obs_max": float(obs_max),
            "obs_min": float(obs_min),
        }
        return p


class neuron_all_trials(): # d'après (Van der Plas 2025)
    def __init__(self, patient, session, ind_neuron, window=(-10, 10), dead_val=np.nan):
        self.patient = patient
        self.session = session[-1]
        self.ind_neuron = ind_neuron
        self.trials = []

        stims = fct.get_stims(patient, session, root)
        for ind_stim in range(stims.shape[0]):
            tr = trial_neuron_stim(self.patient, self.session, self.ind_neuron, ind_stim, window)
            tr.psth(dead_value=dead_val)
            tr.smooth_GC_firing_rate()
            tr.compute_zscore_curot()
            self.trials.append(tr)
            
    def compute_baseline_stats(self, baseline_window=(-10, 0)):
        """
        Calcule la moyenne et l'écart-type de la baseline (pré-stim) 
        sur toutes les FR lissées de tous les trials d'un neurone.
        """
        all_baselines = []        
        for tr in self.trials: # on collecte toutes les baseline
            time_bins, _, _ = tr.psth(bin_size=tr.bin_size)
            mask_baseline = (time_bins >= baseline_window[0]) & (time_bins < baseline_window[1])
            baseline_values = tr.fr_smooth_GC[mask_baseline]
            baseline_values = baseline_values[~np.isnan(baseline_values)]
            all_baselines.append(baseline_values)
        all_baselines = np.concatenate(all_baselines)
        self.baseline_mean = np.nanmean(all_baselines)
        self.baseline_std = np.nanstd(all_baselines)

        print(f"→ Baseline neuron {self.ind_neuron}: mean={self.baseline_mean:.3f}, std={self.baseline_std:.3f}")
        return self.baseline_mean, self.baseline_std

    def normalize_trials(self, min_duration_response = 0.1, lower_thresh = -5, upper_thresh = 5):
        # obsolete, on part sur bootstrap finalement
        """
        Normalise les FR de chaque trial selon la baseline moyenne neuronale
        """

        for tr in self.trials:
            min_bins = int(min_duration_response / tr.bin_size)  # 100 ms minimum
            time_bins = tr.time_bins
            # Normalisation par la baseline moyenne neuronale
            fr_norm = (tr.fr_smooth_GC - self.baseline_mean) / self.baseline_std # pour dead_val = np.nan
            tr.fr_smooth_GC_norm = fr_norm
            fr_norm_dead_zero = (tr.fr_smooth_GC_dead_zero - self.baseline_mean) / self.baseline_std # pour dead_val = 0
            tr.fr_smooth_GC_norm_dead_zero = fr_norm_dead_zero
    
    def run_bootstrap_all_trials(
        self,
        baseline_window=(-5.0, -1.0),
        response_window=(0.0, 4.0),
        n_boot=2000,
        min_duration=0.10,
        block_size=0.05,
        alpha=0.05,
        tail='both',
        use_dead_zero=False
    ):
        """
        Lance le bootstrap FRz(t) sur tous les trials de ce neurone.
        """
        for tr in self.trials:
            if hasattr(tr, 'fr_smooth_GC_norm'):
                tr.bootstrap_responsiveness(
                    baseline_window=baseline_window,
                    response_window=response_window,
                    n_boot=n_boot,
                    min_duration=min_duration,
                    block_size=block_size,
                    alpha=alpha,
                    tail=tail,
                    use_dead_zero=use_dead_zero
                )
            else:
                print('Trial normalisation has not been performed')


patient = 'P115_HO67'
session = '4'
trial_P64_stim2_nrn3_stim10 = trial_neuron_stim(patient, session, 3, 1, window=(-10, 10))
# nrn_P64_stim2_nrn3 = neuron_all_trials(patient, session, 3, window=(-5, 5))
# nrn_P64_stim2_nrn3.compute_baseline_stats()
# nrn_P64_stim2_nrn3.normalize_and_detect_responsiveness()

D:/Spike-sorting/Data_folders/P115_HO67/P115_HO67_stim4/
